# Gereksiz sütunları temizle

Önceki adımda (`1_data_preprocess.ipynb`) ham JSON verisi işlenip
`endomondoHR_proper.csv` olarak kaydedilmişti. Bu notebook'un işi çok
basit ama önemli: CSV'yi tekrar açıp, modelleme için hiçbir değer
katmayan sütunları ayıklamak. Amaç, ilerideki her notebook'un daha az
ve daha anlamlı sütunla, daha küçük bir dosyayla çalışmasını sağlamak.

In [ ]:
import pandas as pd
import numpy as np


In [ ]:
PATH = "/home/can/zero-to-ai-architect/runsight/endomondoHR_proper.csv"
df = pd.read_csv(PATH)

## Önce mevcut hâline bak

Neyi silmemiz gerektiğine karar vermeden önce, elimizdeki sütunları ve
ilk birkaç satırı gözle inceliyoruz. Burada dikkat çeken iki şey var:

- **`Unnamed: 0`** — bu, pandas'ın kendi ürettiği bir sütun değil, önceki
  notebook'ta `df.to_csv(...)` çağrılırken `index=False` YAZILMADIĞI
  için, DataFrame'in satır index'i (0, 1, 2, ...) da CSV'ye ayrı bir
  sütun olarak yazılmış. CSV'yi tekrar `pd.read_csv` ile açınca, pandas
  bu index sütununu tanımadığı için `Unnamed: 0` adıyla sıradan bir veri
  sütunu gibi geri okuyor. Hiçbir analitik değeri yok, sadece kaza eseri
  oluşmuş bir kopya sütun.
- **`url`** — her antrenmanın Endomondo web sitesindeki linki. Modelleme
  açısından (hız, nabız, konum, süre gibi sayısal/zaman serisi
  özelliklere dayanan bir sınıflandırma için) hiçbir bilgi taşımıyor,
  sadece yer kaplıyor.

In [4]:
df.head(10)

,Unnamed: 0,longitude,altitude,latitude,sport,id,heart_rate,gender,url,userId,timestamp,speed
0,0,"[6.8854929, 6.8853678, 6.8851621, 6.8848205, 6...","[-173.8, -151.2, -161.6, -165.4, -168.6, -172....","[52.2226809, 52.222727, 52.2228258, 52.2228606...",run,321063199,"[80.0, 81.0, 94.0, 100.0, 102.0, 112.0, 108.0,...",male,https://www.endomondo.com/users/4969375/workou...,4969375,"[1397079203.0, 1397079210.0, 1397079218.0, 139...",[]
1,1,"[6.9144073, 6.9142929, 6.9141539, 6.9140268, 6...","[57.8, 57.6, 57.0, 56.4, 55.8, 55.2, 54.4, 53....","[52.2111711, 52.2112631, 52.2114064, 52.211608...",run,303565793,"[60.0, 62.0, 92.0, 92.0, 132.0, 150.0, 150.0, ...",male,https://www.endomondo.com/users/4969375/workou...,4969375,"[1393908533.0, 1393908541.0, 1393908549.0, 139...",[]
2,2,"[6.9141348, 6.9145702, 6.9151684, 6.9158377, 6...","[22.8, 26.4, 30.8, 35.6, 43.0, 48.4, 49.8, 49....","[52.2110297, 52.2106325, 52.2102453, 52.209833...",run,302666522,"[77.0, 93.0, 107.0, 121.0, 118.0, 120.0, 120.0...",male,https://www.endomondo.com/users/4969375/workou...,4969375,"[1393687929.0, 1393687948.0, 1393687967.0, 139...",[]
3,3,"[6.8678543, 6.8678634, 6.8675429, 6.8672183, 6...","[35.4, 35.2, 34.6, 34.2, 35.0, 35.2, 34.8, 34....","[52.1936673, 52.1934354, 52.1931993, 52.192873...",run,296982347,"[75.0, 101.0, 116.0, 120.0, 124.0, 126.0, 127....",male,https://www.endomondo.com/users/4969375/workou...,4969375,"[1392480163.0, 1392480176.0, 1392480189.0, 139...",[]
4,4,"[6.9143328, 6.9146396, 6.9148949, 6.9151568, 6...","[63.0, 65.2, 66.0, 66.2, 65.8, 65.8, 67.0, 67....","[52.2112195, 52.2110264, 52.2108135, 52.210601...",run,295890426,"[58.0, 83.0, 112.0, 115.0, 117.0, 116.0, 141.0...",male,https://www.endomondo.com/users/4969375/workou...,4969375,"[1392180426.0, 1392180436.0, 1392180446.0, 139...",[]
5,5,"[6.9133737, 6.9132722, 6.913217, 6.9131066, 6....","[61.4, 56.6, 53.2, 49.6, 46.0, 42.4, 38.8, 35....","[52.2111481, 52.2111209, 52.21119, 52.2112981,...",run,294163731,"[56.0, 62.0, 75.0, 80.0, 80.0, 96.0, 107.0, 10...",male,https://www.endomondo.com/users/4969375/workou...,4969375,"[1391663545.0, 1391663551.0, 1391663557.0, 139...",[]
6,6,"[6.9143129, 6.9145349, 6.9148184, 6.9150409, 6...","[37.8, 41.0, 45.6, 49.8, 53.2, 55.4, 56.4, 57....","[52.2111104, 52.2109796, 52.2108156, 52.210632...",run,293171954,"[58.0, 66.0, 77.0, 95.0, 103.0, 107.0, 111.0, ...",male,https://www.endomondo.com/users/4969375/workou...,4969375,"[1391490093.0, 1391490102.0, 1391490111.0, 139...",[]
7,7,"[6.9144751, 6.9145674, 6.9144764, 6.9142538, 6...","[119.8, 118.0, 115.0, 111.4, 107.8, 104.2, 100...","[52.2110612, 52.2110088, 52.2110491, 52.211245...",run,291223914,"[56.0, 58.0, 61.0, 76.0, 92.0, 99.0, 102.0, 10...",male,https://www.endomondo.com/users/4969375/workou...,4969375,"[1390968925.0, 1390968934.0, 1390968943.0, 139...",[]
8,8,"[6.9144667, 6.9145204, 6.9146491, 6.914725, 6....","[54.6, 54.8, 55.0, 54.8, 54.6, 54.6, 54.6, 54....","[52.2111263, 52.2110629, 52.2109121, 52.210830...",run,288597062,"[62.0, 64.0, 77.0, 88.0, 108.0, 126.0, 134.0, ...",male,https://www.endomondo.com/users/4969375/workou...,4969375,"[1390277966.0, 1390277970.0, 1390277976.0, 139...",[]
9,9,"[6.9144635, 6.9146638, 6.9150391, 6.9146965, 6...","[51.6, 53.2, 57.2, 58.8, 57.6, 57.0, 57.2, 57....","[52.2111823, 52.2109419, 52.2106749, 52.210444...",run,287785413,"[84.0, 94.0, 108.0, 113.0, 121.0, 122.0, 124.0...",male,https://www.endomondo.com/users/4969375/workou...,4969375,"[1390059133.0, 1390059142.0, 1390059153.0, 139...",[]


In [6]:
df.columns

Index(['Unnamed: 0', 'longitude', 'altitude', 'latitude', 'sport', 'id',
       'heart_rate', 'gender', 'url', 'userId', 'timestamp', 'speed'],
      dtype='str')

## İkisini de kaldır

`df.columns` çıktısı, şüphelendiğimiz iki sütunun (`Unnamed: 0`, `url`)
gerçekten orada olduğunu doğruluyor. İkisini de `drop` ile çıkarıyoruz —
geri kalan sütunlara (konum, yükseklik, nabız, hız, zaman, kullanıcı
bilgileri) dokunmuyoruz, onlar sonraki adımlarda (QC, özellik çıkarımı)
kullanılacak.

In [7]:
df = df.drop(columns=['Unnamed: 0','url'])

In [8]:
df.head(10)

,longitude,altitude,latitude,sport,id,heart_rate,gender,userId,timestamp,speed
0,"[6.8854929, 6.8853678, 6.8851621, 6.8848205, 6...","[-173.8, -151.2, -161.6, -165.4, -168.6, -172....","[52.2226809, 52.222727, 52.2228258, 52.2228606...",run,321063199,"[80.0, 81.0, 94.0, 100.0, 102.0, 112.0, 108.0,...",male,4969375,"[1397079203.0, 1397079210.0, 1397079218.0, 139...",[]
1,"[6.9144073, 6.9142929, 6.9141539, 6.9140268, 6...","[57.8, 57.6, 57.0, 56.4, 55.8, 55.2, 54.4, 53....","[52.2111711, 52.2112631, 52.2114064, 52.211608...",run,303565793,"[60.0, 62.0, 92.0, 92.0, 132.0, 150.0, 150.0, ...",male,4969375,"[1393908533.0, 1393908541.0, 1393908549.0, 139...",[]
2,"[6.9141348, 6.9145702, 6.9151684, 6.9158377, 6...","[22.8, 26.4, 30.8, 35.6, 43.0, 48.4, 49.8, 49....","[52.2110297, 52.2106325, 52.2102453, 52.209833...",run,302666522,"[77.0, 93.0, 107.0, 121.0, 118.0, 120.0, 120.0...",male,4969375,"[1393687929.0, 1393687948.0, 1393687967.0, 139...",[]
3,"[6.8678543, 6.8678634, 6.8675429, 6.8672183, 6...","[35.4, 35.2, 34.6, 34.2, 35.0, 35.2, 34.8, 34....","[52.1936673, 52.1934354, 52.1931993, 52.192873...",run,296982347,"[75.0, 101.0, 116.0, 120.0, 124.0, 126.0, 127....",male,4969375,"[1392480163.0, 1392480176.0, 1392480189.0, 139...",[]
4,"[6.9143328, 6.9146396, 6.9148949, 6.9151568, 6...","[63.0, 65.2, 66.0, 66.2, 65.8, 65.8, 67.0, 67....","[52.2112195, 52.2110264, 52.2108135, 52.210601...",run,295890426,"[58.0, 83.0, 112.0, 115.0, 117.0, 116.0, 141.0...",male,4969375,"[1392180426.0, 1392180436.0, 1392180446.0, 139...",[]
5,"[6.9133737, 6.9132722, 6.913217, 6.9131066, 6....","[61.4, 56.6, 53.2, 49.6, 46.0, 42.4, 38.8, 35....","[52.2111481, 52.2111209, 52.21119, 52.2112981,...",run,294163731,"[56.0, 62.0, 75.0, 80.0, 80.0, 96.0, 107.0, 10...",male,4969375,"[1391663545.0, 1391663551.0, 1391663557.0, 139...",[]
6,"[6.9143129, 6.9145349, 6.9148184, 6.9150409, 6...","[37.8, 41.0, 45.6, 49.8, 53.2, 55.4, 56.4, 57....","[52.2111104, 52.2109796, 52.2108156, 52.210632...",run,293171954,"[58.0, 66.0, 77.0, 95.0, 103.0, 107.0, 111.0, ...",male,4969375,"[1391490093.0, 1391490102.0, 1391490111.0, 139...",[]
7,"[6.9144751, 6.9145674, 6.9144764, 6.9142538, 6...","[119.8, 118.0, 115.0, 111.4, 107.8, 104.2, 100...","[52.2110612, 52.2110088, 52.2110491, 52.211245...",run,291223914,"[56.0, 58.0, 61.0, 76.0, 92.0, 99.0, 102.0, 10...",male,4969375,"[1390968925.0, 1390968934.0, 1390968943.0, 139...",[]
8,"[6.9144667, 6.9145204, 6.9146491, 6.914725, 6....","[54.6, 54.8, 55.0, 54.8, 54.6, 54.6, 54.6, 54....","[52.2111263, 52.2110629, 52.2109121, 52.210830...",run,288597062,"[62.0, 64.0, 77.0, 88.0, 108.0, 126.0, 134.0, ...",male,4969375,"[1390277966.0, 1390277970.0, 1390277976.0, 139...",[]
9,"[6.9144635, 6.9146638, 6.9150391, 6.9146965, 6...","[51.6, 53.2, 57.2, 58.8, 57.6, 57.0, 57.2, 57....","[52.2111823, 52.2109419, 52.2106749, 52.210444...",run,287785413,"[84.0, 94.0, 108.0, 113.0, 121.0, 122.0, 124.0...",male,4969375,"[1390059133.0, 1390059142.0, 1390059153.0, 139...",[]


## Sonucu kaydet — bu sefer `index=False` ile

Aynı `endomondoHR_proper.csv` dosyasının üzerine yazıyoruz, ama bu kez
`index=False` parametresini kullanıyoruz — böylece pandas satır
index'ini ayrı bir sütun olarak CSV'ye yazmıyor. Bu, en başta gördüğümüz
`Unnamed: 0` sorununun tekrar oluşmasını engelliyor: bu dosyayı bundan
sonra kim okursa okusun, artık kaza eseri bir index sütunuyla
karşılaşmayacak.

In [9]:
df.to_csv("/home/can/zero-to-ai-architect/runsight/endomondoHR_proper.csv", index=False)